# VECM Cointegration & Macro-Yield Curve Dynamics

Estimates the cointegrated `VECM` specification in levels (`beta0`, `beta1`, `beta2`, `cpi_mom`) using Johansen trace/max-eigenvalue tests, Granger causality, impulse response functions, and `VAR` robustness checks in first differences.

In [ ]:
import json
import warnings
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from ofz_curve.paths import FIXED_TAU1_FACTORS_CSV, RUSSIA_CPI_CSV

warnings.filterwarnings('ignore')

In [ ]:
# 1. Подготовка данных в единой месячной частоте (MS)
nss = pd.read_csv(FIXED_TAU1_FACTORS_CSV)
cpi = pd.read_csv(RUSSIA_CPI_CSV)

nss['date'] = pd.to_datetime(nss['date'], errors='coerce')
cpi['date'] = pd.to_datetime(cpi['date'], errors='coerce')

nss = nss.dropna(subset=['date']).set_index('date').sort_index()
cpi = cpi.dropna(subset=['date']).set_index('date').sort_index()

nss_ms = nss[['beta0', 'beta1', 'beta2']].resample('MS').last()
cpi_ms = cpi[['cpi_mom', 'cpi_yoy']].copy()
cpi_ms.index = cpi_ms.index.to_period('M').to_timestamp(how='start')

var_data = nss_ms.join(cpi_ms, how='inner').dropna()
levels_data = var_data[['beta0', 'beta1', 'beta2', 'cpi_mom']].copy()

print(f'Период: {levels_data.index.min().date()} .. {levels_data.index.max().date()}')
print(f'Наблюдений: {len(levels_data)}')
display(levels_data.head())

In [ ]:
# 1b. Расширенная описательная визуализация (возвращено из прежней версии)
display(levels_data.describe())

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
levels_data['beta0'].plot(ax=axes[0,0], title='beta0 (уровень)', color='blue')
levels_data['beta1'].plot(ax=axes[0,1], title='beta1 (наклон)', color='red')
levels_data['beta2'].plot(ax=axes[1,0], title='beta2 (кривизна)', color='green')
levels_data['cpi_mom'].plot(ax=axes[1,1], title='cpi_mom (м/м)', color='orange')
var_data['cpi_yoy'].plot(ax=axes[2,0], title='cpi_yoy (г/г)', color='purple')
sns.heatmap(levels_data.corr(), annot=True, cmap='coolwarm', center=0, ax=axes[2,1], fmt='.2f')
axes[2,1].set_title('Корреляционная матрица (уровни)')
for ax in axes.flat:
    if ax is not axes[2,1]:
        ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Диагностический ADF (не для выбора основной модели)
adf_rows = []
for col in levels_data.columns:
    stat, pval, _, _, _, _ = adfuller(levels_data[col].dropna(), autolag='AIC')
    adf_rows.append({'variable': col, 'adf_stat': stat, 'p_value': pval, 'stationary_5pct': pval <= 0.05})
adf_df = pd.DataFrame(adf_rows)
display(adf_df)

In [ ]:
# 3. Выбор лага VAR в уровнях и тест Йохансена
lag_res = VAR(levels_data).select_order(maxlags=12)
display(lag_res.summary())
p_levels = max(1, int(lag_res.aic) if lag_res.aic is not None else 1)
k_ar_diff = max(0, p_levels - 1)

jres = coint_johansen(levels_data, det_order=0, k_ar_diff=k_ar_diff)
trace = pd.DataFrame({
    'r<=k': range(len(jres.lr1)),
    'trace_stat': jres.lr1,
    'crit95': jres.cvt[:, 1],
})
trace['reject_95'] = trace['trace_stat'] > trace['crit95']

display(trace)

maxeig = pd.DataFrame({
    'r=k': range(len(jres.lr2)),
    'maxeig_stat': jres.lr2,
    'crit95': jres.cvm[:, 1],
})
maxeig['reject_95'] = maxeig['maxeig_stat'] > maxeig['crit95']

display(maxeig)

coint_rank = 0
for _, row in trace.iterrows():
    if row['reject_95']:
        coint_rank = int(row['r<=k']) + 1
    else:
        break
coint_rank = max(1, min(coint_rank, levels_data.shape[1] - 1))

print(f'p_levels={p_levels}, k_ar_diff={k_ar_diff}, coint_rank={coint_rank}')


In [ ]:
# 4. Основная модель: VECM в уровнях
vecm = VECM(levels_data, k_ar_diff=k_ar_diff, coint_rank=coint_rank, deterministic='co')
vecm_res = vecm.fit()

alpha_df = pd.DataFrame(vecm_res.alpha, index=levels_data.columns, columns=[f'ec{i+1}' for i in range(coint_rank)])
beta_df = pd.DataFrame(vecm_res.beta, index=levels_data.columns, columns=[f'ec{i+1}' for i in range(coint_rank)])

print('Коэффициенты корректировки ошибок (alpha):')
display(alpha_df)
print('Коинтеграционные соотношения (beta):')
display(beta_df)

In [ ]:
# 5. Диагностика остатков VECM
resid = pd.DataFrame(vecm_res.resid, columns=levels_data.columns)
lb_rows = []
for col in resid.columns:
    lb = acorr_ljungbox(resid[col], lags=[10], return_df=True)
    pval = float(lb['lb_pvalue'].iloc[-1])
    lb_rows.append({'variable': col, 'lb_pvalue_lag10': pval, 'no_autocorr_5pct': pval > 0.05})
lb_df = pd.DataFrame(lb_rows)
display(lb_df)

In [ ]:
# 6. IRF на основе VECM (основной результат)
irf = vecm_res.irf(24)
cols = list(levels_data.columns)
n = len(cols)
fig, axes = plt.subplots(n, n, figsize=(15, 11), sharex=True)
x = np.arange(irf.irfs.shape[0])
for i in range(n):
    for j in range(n):
        ax = axes[i, j]
        ax.plot(x, irf.irfs[:, i, j], color='#1f77b4')
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
        ax.set_title(f'{cols[j]} -> {cols[i]}', fontsize=9)
        ax.grid(alpha=0.2)
plt.suptitle('IRF на основе VECM (уровни)', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
# 7. FEVD: fallback через VAR(levels) для сопоставимости
var_fit_levels = VAR(levels_data).fit(maxlags=p_levels, trend='c')
fevd = var_fit_levels.fevd(24)
cpi_idx = cols.index('cpi_mom')
rows = []
for h in [1, 6, 12, 24]:
    for shock_idx, shock in enumerate(cols):
        rows.append({'horizon': h, 'response': 'cpi_mom', 'shock': shock, 'share_pct': float(fevd.decomp[cpi_idx, h-1, shock_idx] * 100)})
fevd_df = pd.DataFrame(rows)
display(fevd_df)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, h in zip(axes.flatten(), [1, 6, 12, 24]):
    cur = fevd_df[fevd_df['horizon'] == h]
    ax.pie(cur['share_pct'], labels=cur['shock'], autopct='%1.1f%%', startangle=90)
    ax.set_title(f'FEVD для cpi_mom, горизонт {h} мес.')
plt.tight_layout()
plt.show()

In [ ]:
# 7b. Детализированный FEVD (таблицы + накопительные диаграммы)
horizons_to_display = [1, 6, 12, 24]
for h in horizons_to_display:
    cur = fevd_df[fevd_df['horizon'] == h].copy()
    cur = cur[['shock', 'share_pct']].rename(columns={'share_pct': f'h{h}_share_pct'})
    print(f'FEVD cpi_mom, горизонт {h} мес:')
    display(cur)

# накопительная динамика вкладов по горизонтам
horizons = sorted(fevd_df['horizon'].unique())
pivot = fevd_df.pivot(index='horizon', columns='shock', values='share_pct').reindex(horizons)
fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(len(pivot))
for col in pivot.columns:
    vals = pivot[col].values
    ax.bar(pivot.index, vals, bottom=bottom, label=col)
    bottom += vals
ax.set_title('FEVD для cpi_mom по горизонтам (fallback VAR-levels)')
ax.set_xlabel('горизонт, мес.')
ax.set_ylabel('доля дисперсии, %')
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 8. Granger: основной (уровни) и проверка устойчивости (разности)
def granger_table(df, lag):
    rows = []
    lag = max(1, min(int(lag), 4))
    for target in df.columns:
        for source in df.columns:
            if source == target:
                continue
            pair = df[[target, source]].dropna()
            g = grangercausalitytests(pair, maxlag=lag, verbose=False)
            pval = float(g[lag][0]['ssr_ftest'][1])
            rows.append({'direction': f'{source} -> {target}', 'lag': lag, 'p_value': pval, 'significant_5pct': pval < 0.05})
    return pd.DataFrame(rows).sort_values('p_value').reset_index(drop=True)

granger_levels = granger_table(levels_data, p_levels)
diff_data = levels_data.diff().dropna()
diff_data.columns = [f'd_{c}' for c in diff_data.columns]
granger_diff = granger_table(diff_data, p_levels)

print('Granger (levels, основной):')
display(granger_levels)
print('Granger (differences, проверка устойчивости):')
display(granger_diff)

In [ ]:
# 8b. Экономическая интерпретация (возвращено в релевантном виде)
print('ЭКОНОМИЧЕСКАЯ ИНТЕРПРЕТАЦИЯ')
print('='*60)

sig_levels = granger_levels[granger_levels['significant_5pct']]
print('1) Granger (уровни), значимые направления:')
if len(sig_levels) == 0:
    print('   значимых направлений нет на 5%')
else:
    for _, r in sig_levels.iterrows():
        print(f'   - {r["direction"]}: p={r["p_value"]:.4f}')

print('\n2) FEVD для cpi_mom (основные горизонты):')
for h in [1, 6, 12, 24]:
    cur = fevd_df[fevd_df['horizon'] == h].sort_values('share_pct', ascending=False)
    top = cur.iloc[0]
    curve_share = cur[cur['shock'] != 'cpi_mom']['share_pct'].sum()
    print(f'   h={h}: ведущий шок {top["shock"]} ({top["share_pct"]:.2f}%), вклад факторов кривой={curve_share:.2f}%')

print('\n3) Сопоставление с проверкой устойчивости в разностях:')
sig_diff = granger_diff[granger_diff['significant_5pct']]
if len(sig_diff) == 0:
    print('   в разностях значимых направлений на 5% нет')
else:
    for _, r in sig_diff.iterrows():
        print(f'   - {r["direction"]}: p={r["p_value"]:.4f}')

In [ ]:
# 9. Экспорт артефактов для LaTeX-приложений
out_dir = ROOT / 'data' / 'processed' / 'vecm_outputs'
img_dir = ROOT / 'writing' / 'latex' / 'media' / 'media'
out_dir.mkdir(parents=True, exist_ok=True)
img_dir.mkdir(parents=True, exist_ok=True)

# Таблицы
pd.DataFrame(lag_res.summary().data[1:], columns=lag_res.summary().data[0]).to_csv(out_dir / 'lag_selection_levels.csv', index=False)
trace.to_csv(out_dir / 'johansen_trace.csv', index=False)
maxeig.to_csv(out_dir / 'johansen_maxeig.csv', index=False)
alpha_df.to_csv(out_dir / 'vecm_alpha.csv')
beta_df.to_csv(out_dir / 'vecm_beta.csv')
lb_df.to_csv(out_dir / 'vecm_ljungbox.csv', index=False)
granger_levels.to_csv(out_dir / 'granger_levels.csv', index=False)
granger_diff.to_csv(out_dir / 'granger_diff.csv', index=False)
fevd_df.to_csv(out_dir / 'fevd_cpi_mom.csv', index=False)

summary = {
    'n_levels_obs': int(len(levels_data)),
    'period_start': str(levels_data.index.min().date()),
    'period_end': str(levels_data.index.max().date()),
    'p_levels_aic': int(p_levels),
    'k_ar_diff': int(k_ar_diff),
    'coint_rank_trace_95': int(coint_rank),
    'fevd_source': 'VAR(levels) fallback'
}
(out_dir / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

# image9
fig9, axes9 = plt.subplots(3, 2, figsize=(14, 11))
var_data['beta0'].plot(ax=axes9[0,0], title='beta0 (уровень)', color='#1f77b4')
var_data['beta1'].plot(ax=axes9[0,1], title='beta1 (наклон)', color='#d62728')
var_data['beta2'].plot(ax=axes9[1,0], title='beta2 (кривизна)', color='#2ca02c')
var_data['cpi_mom'].plot(ax=axes9[1,1], title='Инфляция cpi_mom (м/м)', color='#ff7f0e')
var_data['cpi_yoy'].plot(ax=axes9[2,0], title='Инфляция cpi_yoy (г/г)', color='#9467bd')
sns.heatmap(var_data[['beta0','beta1','beta2','cpi_mom']].corr(), annot=True, cmap='coolwarm', center=0, ax=axes9[2,1], fmt='.2f')
axes9[2,1].set_title('Корреляционная матрица')
for ax in axes9.flat:
    if ax is not axes9[2,1]:
        ax.grid(alpha=0.3)
fig9.tight_layout()
fig9.savefig(img_dir / 'image9.png', dpi=220)
plt.close(fig9)

# image10
fig10, axes10 = plt.subplots(n, n, figsize=(15, 11), sharex=True)
x = np.arange(irf.irfs.shape[0])
for i in range(n):
    for j in range(n):
        ax = axes10[i, j]
        ax.plot(x, irf.irfs[:, i, j], color='#1f77b4')
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
        ax.set_title(f'{cols[j]} -> {cols[i]}', fontsize=9)
        ax.grid(alpha=0.2)
fig10.suptitle('IRF на основе VECM (уровни)', fontsize=14)
fig10.tight_layout(rect=[0, 0, 1, 0.97])
fig10.savefig(img_dir / 'image10.png', dpi=220)
plt.close(fig10)

# image11
fig11, axes11 = plt.subplots(2, 2, figsize=(12, 9))
for ax, h in zip(axes11.flatten(), [1, 6, 12, 24]):
    cur = fevd_df[fevd_df['horizon'] == h]
    ax.pie(cur['share_pct'], labels=cur['shock'], autopct='%1.1f%%', startangle=90)
    ax.set_title(f'FEVD для cpi_mom, горизонт {h} мес.')
fig11.tight_layout()
fig11.savefig(img_dir / 'image11.png', dpi=220)
plt.close(fig11)

print('Экспорт завершён:', out_dir)
